In [20]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
# input = r"/Users/cmdb/Quant_Bio_Project/Quant-Bio-Project/segmentation_test.tif" 
# img = cv.imread(input, cv.IMREAD_GRAYSCALE)



In [21]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

def gaussian_kernal(size,std):
    kernel = np.fromfunction(
        lambda x,y: np.divide(1,2*np.pi* std**2) * 
        np.exp(
            -((x-(size-1)/2)**2 + (y-(size-1)/2)**2) / (2* std**2)
               ),
        (size,size)
    )
    return np.array(kernel/np.sum(kernel))

def find(frame,mask,contour,pixelSize):

    """
    This function takes in an image and draws the given contours.  It returns an inverted boolean array
    where true = pixel within a contour. This gives the number of pixels within a given contour to calculate area, as well as the 
    starting point for blob labeling and tracking.
    """
    #dont want to edit the original image
    filled_mask = np.copy(mask)
    #draw the contour
    filled_mask = cv.drawContours(filled_mask,[contour], 0,(0, 255, 0),thickness=cv.FILLED)
    #boolean mask and returning it, ensuring true = 0, false = 1. The sum of this is pixel size. 
    image_mask = filled_mask > 0
    total_pixels = np.sum(~image_mask.astype(int))
    image_mask = ~image_mask
    total_pixels = np.sum(image_mask.astype(int))
    #output [frame, x,y, Area, TotalIntensity]
    total_intensity = np.sum(mask*(image_mask.astype(int)))
    ##Find X,Y center of mass based on contours. 
    #https://docs.opencv.org/3.4/dd/d49/tutorial_py_contour_features.html
    M = cv.moments(contour)
    x = int(M['m10']/M['m00'])
    y = int(M['m01']/M['m00'])
    #in units of pixelSize(micron squared)
    total_area = total_pixels*(pixelSize**2)
    return [frame,[x,y],total_area,total_intensity]
kernel = gaussian_kernal(3,np.sqrt(3))
low_thresh: int = 13
high_thresh: int = 50

#contour filtering
# we want anything equal to or greater than 5
filter_val = 10

##Pixel Size. This is used for area calculations. Where the area of a pixel is pixelSize**2
pixel_size: float = 0.115 


## Apply pixel normalization

Min Pixel Value = min(image)
Max Pixel Value = max(image)

Normalized = {pixel - min pixel}/{max pixel - min pixel}

In [22]:
# contrs, hierch = cv.findContours(thresh,1,2)

Next? Check contrs output

In [23]:
# epsilon = cv.arcLength(cnt,True)
# approx = cv.approxPolyDP(cnt,epsilon,True)

## Time to work on the full stack analysis

#### Working with a stacked tiff

Implemented

## Convert from uint16 to uint8
CV2 threshold function only works with uint8 values, so we have to convert it. This is done by dividing it by 256 and assigning it as a uint8 datatype. 

## Histogram Normalization

In [24]:
# image1 = cv.filter2D(tiff_stack[10],-1,kernel)
# plt.imshow(image1)
# plt.show()

Converted stacked tiff into opencv 

In [25]:
import skimage.io as io
tiff_stack = io.imread("/Users/cmdb/Quant_Bio_Project/Quant-Bio-Project/cell2.tif",plugin='tifffile')
tiff_stack = (tiff_stack/256).astype(np.uint8)
copied = np.copy(tiff_stack)
frame_dict = {}
for i in range(len(tiff_stack)):
    ret,thresh = cv.threshold(tiff_stack[i,:,:],low_thresh,high_thresh,cv.THRESH_BINARY)
    contours,hierarchy = cv.findContours(thresh, 1, 2)
    #filtering out contours less than 10
    contours = [lst for lst in contours if len(lst) >= filter_val]
    frame_dict[f"Timepoint {i}"] = []
    for j in contours:
        #thresholding step needed to skip bad areas. Area == 0 is nothing. 
        area = cv.moments(j)["m00"]
        if area > 0:
            centroid_output = find(i,copied[i,:,:],j,pixel_size)
            frame_dict[f"Timepoint {i}"].append(centroid_output)


## Tracking by Elucidian Distance

In [26]:
##Challenges
# iterate through the dictionary by frame ##### success
# iterate through the dictionary by frame and frame + 1 ##### success
# pull x,y coords ##### success
# create blobs for n ##### success
# # pair n+1 with n blobs (first two frames only)

def elucidian_distance(n_coords,n1_coords):
   # based on 2d distance formula of sqrt((x2-x1)**2 + (y2-y1)**2))
   x1 = n_coords[0]
   y1 = n_coords[1]
   x2 = n1_coords[0]
   y2 = n1_coords[1]
   distance = np.sqrt((x2-x1)**2 + (y2-y1)**2)
   return distance

def search_matching_by_distance(blobs,n_coords,n1_coords,thresh):

   
   """
   take a single coord from n: this is the input n_coords <- incorrect, you might pair previous coords with n+1 coords. 
   iterate through all n+1, this is all n+1 coords
   find coords within the threshold distance
   if within threshold distance, assign n+1 to that blob
   if no blob is found within threshold distance
   if multiple blobs found within threshold distance
         find a way to score it?
         cross that bridge when we have to 
   create new blob 
   repeat 

   """
   #dictionary with current blobs
   blob_dict = blobs
   distances = []
   for i in n1_coords:
      #first [1] is coords
      #second [] is x,y where x = 0, y = 1
      distance = elucidian_distance(n_coords[1],i[1])
   pass

def blob_tracking_simple(frame,dict,n,n1,threshold):
   pairs = []
   #input n is all blobs in frame
   #n[i] denotes the individual blobs
   for i in range(len(n)):
      #this assigns the first frame centroids as blobs i+1. these are the first blobs 
      if frame == 0:
         dict[f"Blob {i+1}"] = [n[i]]
      #this function takes in a single blob from n and compares it to all n1 coords
      search_matching_by_distance(dict,n_coords=n[i],n1_coords=n1,thresh=threshold)
      for j in range(len(n1)):

         distance = elucidian_distance(n_coords=n[i][1],n1_coords=n1[j][1])
         if distance < threshold:
            pass
      print(i)


   pass
   
#keys list
list_of_keys = sorted(frame_dict.keys())
#this is where we will be storing the tracked blobs
blobs: dict = {}
for i in range(1):
   n = list_of_keys[i]
   n1 = list_of_keys[i+1]
   coords_n = frame_dict[n]
   coords_n1 = frame_dict[n1]
   
   blobs = blob_tracking_simple(i,blobs,coords_n,coords_n1,threshold=5)


   

0
1
2
3
4
5
6
7


## this is going to get really messy

heres the matrix implementation of the hungarian algorithm based on cost-value matrixes. based on the wikipedia walkthrough
https://en.wikipedia.org/wiki/Hungarian_algorithm#Matrix_interpretation

## Step 1, Create the cost matrix
Here the cost between n and n+1 is the distance. 

In [27]:
def test_one_zeros(matrix,xy):
        num_zeros = np.sum(matrix == 0, axis=xy)
        if max(num_zeros) > 1:
            return False
        else:
            return True
def matrix_generation(p,o):
    #where p = n
    #where o = n+1
    top_row = []
    side_row = []
    for i in p:
        top_row.append(i[1])
    for j in o:
        side_row.append(j[1])

    #time to create the matrix
    cost_matrix = []
    for b in top_row:
        output_row = []
        for c in side_row:
            cost = np.sqrt((c[0]-b[0])**2 + (c[1]-b[1])**2)
            output_row.append(cost)
        cost_matrix.append(output_row)

    return(np.array(cost_matrix))
def subtract_minimum(matrix):
    backup_matrix = np.copy(matrix)
    cnt = 0
    for i in range(len(matrix)):
        min = np.min(matrix[i,:])
        matrix[i,:] =  matrix[i,:] - min
        cnt += 1
    if test_one_zeros(matrix,xy=0) == False:
        for k in range((np.shape(backup_matrix)[1])):
            min = np.min(backup_matrix[:,k])
            backup_matrix[:,k] = backup_matrix[:,k] - min
        return backup_matrix
    else:
        return matrix
        

                
for i in range(1):
   n = list_of_keys[i]
   n1 = list_of_keys[i+1]
   coords_n = frame_dict[n]
   coords_n1 = frame_dict[n1]
   cost_matrix = matrix_generation(p=coords_n,o=coords_n1)
cost_matrix = subtract_minimum(cost_matrix)
if test_one_zeros(cost_matrix,xy=0) == False:
    print('cols')
if test_one_zeros(cost_matrix,xy=1) == False:
    print('rows')
## Where there is a row and column with just a single zero, that is what should be assigned to a blob. 
## if there is a row or column with multiple zeros, that is the excess blob and should be added to the dictionary.
#I really hope this works 

def find_minimum_cost(matrix):
    null = np.where(cost_matrix ==0)
    return null
#where we have a cost value of 0
output = np.where(cost_matrix == 0)

"""
    Step 2

    Subtract the minimum from each row in the cost matrix. Each row should have 1 zero
"""

rows


'\n    Step 2\n\n    Subtract the minimum from each row in the cost matrix. Each row should have 1 zero\n'

## step 2

In [31]:

## remember row is a blob
## 0 and 0 are together
## 1 and 1
## 3 an 2
## 4 and 3
## 5 and 5

def blob_pairing(blob_dict, where_output,n,n1):
    ## the goal of this function is the pair the output of np.where and the cost analysis matrix implementation
    #reminders
    ## rows are n+1, cols are n
    ## rows that appear more than once, are new blobs, add them to blob_dict
    ## must recalculate matrix without that row then pair the blobs!
    for j in range(len(where_output[0])):
        n1_index = where_output[0][j]
        n_index = where_output[1][j]
        if n[n_index] not in blob_dict:
            blob_dict[n[n_index]] = [f"Blob {len(blob_dict.keys())+1}"]
            
        else: 

    pass

blob_pairing(blobs,output,coords_n,coords_n1)

[0 1 3 4 5 7 7]
